# putEMG — Full Pipeline

Run in Google Colab.

| Part | Description | Output |
|------|-------------|--------|
| **1** | `.mat` → bandpass/notch/resample/z-score → features `.npz` | `data/processed gestures/`  `data/features/` |
| **2a** | Within-subject 3-fold CV — EEGNet, ShallowConvNet, EMG_TCN, FeatureLSTM, FeatureGRU, FeatureTransformer | `baselines/within_subject/` |
| **2b** | Cross-subject LOSO — same 6 models | `baselines/cross_subject/` |

**How to run a single part independently:**
1. Run the **Install** cell below → restart the session when prompted
2. Jump to the Part you want → run its **Setup** cell, then the remaining cells in that Part

Already-complete subjects / folds are skipped automatically — safe to interrupt and resume.

#### Install — run once per session, then restart

In [4]:
print(4)

4


In [2]:
print(9)
# Run this cell, then: Runtime → Restart session
# After restart, jump to whichever Part you want and run its Setup cell.
!pip install -q torch torchvision torchaudio
!pip install -q libemg                                        # installs with all deps (numba etc.)
!pip install -q --force-reinstall 'numpy==1.26.4' 'scipy<1.15'  # force-pin both so binaries match
!pip install -q pandas==2.2.2 matplotlib

9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible

---
## Part 1 — Data Preparation

Converts raw MATLAB recordings to model-ready feature files.

**Step 1 — Preprocessing:** `combinedCell .mat` → bandpass 20–700 Hz → notch 30/50/60/90/150 Hz → resample 1500 samples → z-score → `X:(N,24,1500)` + `y:(N,)`

**Step 2 — Feature extraction:** sliding window (size=250, shift=50) → 50 libemg features × 24 channels → `X:(N,82992)` flat-per-rep `.npz`

Already-complete subjects are skipped automatically.

In [3]:
# ── Part 1 Setup (mount + paths + imports) ────────────────────────────────────
from google.colab import drive
import os, sys, glob
drive.mount('/content/drive')

_base           = '/content/drive/Othercomputers/My Mac/putEMG prime'
UNPROCESSED_DIR = os.path.join(_base, 'data/unprocessed gestures')
PROCESSED_DIR   = os.path.join(_base, 'data/processed gestures')
TRAIN_DIR       = os.path.join(PROCESSED_DIR, 'train')
EVAL_DIR        = os.path.join(PROCESSED_DIR, 'eval')
FEAT_DIR        = os.path.join(_base, 'data/features')
FEAT_TRAIN      = os.path.join(FEAT_DIR, 'train')
FEAT_EVAL       = os.path.join(FEAT_DIR, 'eval')

for _p in (os.path.join(_base, 'src'), os.path.join(_base, 'data_preprocessing')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from preprocessing      import batch_process_split_dirs
from src.feature_extraction import batch_extract_split_dirs
print('Part 1 setup OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Part 1 setup OK


In [4]:
# ── Step 1 — Preprocess .mat → .npz ──────────────────────────────────────────
batch_process_split_dirs(
    input_root  = UNPROCESSED_DIR,
    output_root = PROCESSED_DIR,
)


  Processing split: train
Found 44 subject file(s) in train/
Bandpass 20–700 Hz + Notch [30, 50, 60, 90, 150] Hz: ON | Z-score: ON | Target length: 1500

[1/44] emg_gestures_03_NU_mat.mat
  Skipping (already exists): emg_gestures_03_U.npz

[2/44] emg_gestures_04_NU_mat.mat
  Skipping (already exists): emg_gestures_04_U.npz

[3/44] emg_gestures_05_NU_mat.mat
  Skipping (already exists): emg_gestures_05_U.npz

[4/44] emg_gestures_06_NU_mat.mat
  Skipping (already exists): emg_gestures_06_U.npz

[5/44] emg_gestures_07_NU_mat.mat
  Skipping (already exists): emg_gestures_07_U.npz

[6/44] emg_gestures_08_NU_mat.mat
  Skipping (already exists): emg_gestures_08_U.npz

[7/44] emg_gestures_09_NU_mat.mat
  Skipping (already exists): emg_gestures_09_U.npz

[8/44] emg_gestures_10_NU_mat.mat
  Skipping (already exists): emg_gestures_10_U.npz

[9/44] emg_gestures_11_NU_mat.mat
  Skipping (already exists): emg_gestures_11_U.npz

[10/44] emg_gestures_12_NU_mat.mat
  Skipping (already exists): emg_ges

In [5]:
# ── Verify Step 1 ─────────────────────────────────────────────────────────────
train_npz = glob.glob(os.path.join(TRAIN_DIR, '*.npz'))
eval_npz  = glob.glob(os.path.join(EVAL_DIR,  '*.npz'))
print(f'Processed train : {len(train_npz)} files  →  {TRAIN_DIR}')
print(f'Processed eval  : {len(eval_npz)}  files  →  {EVAL_DIR}')
if not train_npz or not eval_npz:
    raise RuntimeError('No processed files found — check UNPROCESSED_DIR and re-run Step 1.')

Processed train : 44 files  →  /content/drive/Othercomputers/My Mac/putEMG prime/data/processed gestures/train
Processed eval  : 44  files  →  /content/drive/Othercomputers/My Mac/putEMG prime/data/processed gestures/eval


In [6]:
# ── Step 2 — Extract features ─────────────────────────────────────────────────
batch_extract_split_dirs(
    input_root   = PROCESSED_DIR,
    output_root  = FEAT_DIR,
    window_size  = 250,
    window_shift = 50,
    force        = False,
)


  Extracting features: train
Found 44 file(s) — libemg 50 features × 24 ch, flat_rep

  [SKIP] emg_gestures_03_U.npz
  [SKIP] emg_gestures_04_U.npz
  [SKIP] emg_gestures_05_U.npz
  [SKIP] emg_gestures_06_U.npz
  [SKIP] emg_gestures_07_U.npz
  [SKIP] emg_gestures_08_U.npz
  [SKIP] emg_gestures_09_U.npz
  [SKIP] emg_gestures_10_U.npz
  [SKIP] emg_gestures_11_U.npz
  [SKIP] emg_gestures_12_U.npz
  [SKIP] emg_gestures_13_U.npz
  [SKIP] emg_gestures_14_U.npz
  [SKIP] emg_gestures_15_U.npz
  [SKIP] emg_gestures_16_U.npz
  [SKIP] emg_gestures_17_U.npz
  [SKIP] emg_gestures_18_U.npz
  [SKIP] emg_gestures_19_U.npz
  [SKIP] emg_gestures_20_U.npz
  [SKIP] emg_gestures_22_U.npz
  [SKIP] emg_gestures_23_U.npz
  [SKIP] emg_gestures_24_U.npz
  [SKIP] emg_gestures_25_U.npz
  [SKIP] emg_gestures_26_U.npz
  [SKIP] emg_gestures_27_U.npz
  [SKIP] emg_gestures_29_U.npz
  [SKIP] emg_gestures_30_U.npz
  [SKIP] emg_gestures_31_U.npz
  [SKIP] emg_gestures_33_U.npz
  [SKIP] emg_gestures_34_U.npz
  [SKIP] emg_g

In [7]:
# ── Verify Step 2 ─────────────────────────────────────────────────────────────
feat_train = glob.glob(os.path.join(FEAT_TRAIN, '*.npz'))
feat_eval  = glob.glob(os.path.join(FEAT_EVAL,  '*.npz'))
print(f'Feature train : {len(feat_train)} files  →  {FEAT_TRAIN}')
print(f'Feature eval  : {len(feat_eval)}  files  →  {FEAT_EVAL}')
if not feat_train or not feat_eval:
    raise RuntimeError('No feature files found — run Step 1 first, then re-run Step 2.')

Feature train : 44 files  →  /content/drive/Othercomputers/My Mac/putEMG prime/data/features/train
Feature eval  : 44  files  →  /content/drive/Othercomputers/My Mac/putEMG prime/data/features/eval


---
## Part 2a — Within-Subject Baselines

**Protocol:** stratified 3-fold CV per subject (train on 2 folds, test on 1, rotate 3×).
Data: train + eval sessions combined per subject (~280 reps).

| Model type | Models | Input |
|------------|--------|-------|
| Deep learning | EEGNet, ShallowConvNet, EMG_TCN | `(N, 1, 24, 1500)` |
| Feature-based | **FeatureMLP** (PyTorch) + **SVM** (sklearn) | `(N, 3192)` mean-pooled |

Checkpoints saved per subject-model pair — safe to interrupt and resume.

In [8]:
# ── Part 2a Setup (mount + paths + imports) ───────────────────────────────────
from google.colab import drive
import os, sys, glob, torch
import numpy as np
drive.mount('/content/drive')

_base           = '/content/drive/Othercomputers/My Mac/putEMG prime'
PROCESSED_DIR   = os.path.join(_base, 'data/processed gestures')
TRAIN_DIR       = os.path.join(PROCESSED_DIR, 'train')
EVAL_DIR        = os.path.join(PROCESSED_DIR, 'eval')
FEAT_DIR        = os.path.join(_base, 'data/features')
FEAT_TRAIN      = os.path.join(FEAT_DIR, 'train')
FEAT_EVAL       = os.path.join(FEAT_DIR, 'eval')
WS_DL_WEIGHTS   = os.path.join(_base, 'baselines/within_subject/deep_learning/weights')
WS_DL_RESULTS   = os.path.join(_base, 'baselines/within_subject/deep_learning/results')
WS_FB_WEIGHTS   = os.path.join(_base, 'baselines/within_subject/feature_based/weights')
WS_FB_RESULTS   = os.path.join(_base, 'baselines/within_subject/feature_based/results')

for _p in (os.path.join(_base, 'src'), os.path.join(_base, 'data_preprocessing')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import deep_learning_models as dl_models
import feature_based_models as fb_models
from emg_loader import load_subjects_combined, load_feature_subjects_combined
from trainer    import run_within_subject, run_svm_within_subject

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for _d in (WS_DL_WEIGHTS, WS_DL_RESULTS, WS_FB_WEIGHTS, WS_FB_RESULTS):
    os.makedirs(_d, exist_ok=True)
print(f'Part 2a setup OK  |  device: {device}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Part 2a setup OK  |  device: cuda


In [9]:
# ── Training config ───────────────────────────────────────────────────────────
N_FOLDS    = 3
BATCH_SIZE = 16
SEED       = 42
MAX_EPOCHS = 30
PATIENCE   = 5
MIN_DELTA  = 0.001
LR         = 1e-3
DROPOUT    = 0.1

In [10]:
# ── Deep learning — 3-fold CV ─────────────────────────────────────────────────
dl_subjects = load_subjects_combined(TRAIN_DIR, EVAL_DIR)
dl_registry = {
    'EEGNet':         lambda: dl_models.EEGNet(dropout_rate=DROPOUT),
    'ShallowConvNet': lambda: dl_models.ShallowConvNet(dropout_rate=DROPOUT),
    'EMG_TCN':        lambda: dl_models.EMG_TCN(dropout_rate=DROPOUT),
}
run_within_subject(
    dl_subjects, dl_registry, WS_DL_WEIGHTS, device,
    n_folds=N_FOLDS, batch_size=BATCH_SIZE, seed=SEED,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, lr=LR,
    results_dir=WS_DL_RESULTS,
)

Combining train+eval for 44 subject(s)

  → emg_gestures_03_U.npz  (272 reps)
  → emg_gestures_04_U.npz  (254 reps)
  → emg_gestures_05_U.npz  (273 reps)
  → emg_gestures_06_U.npz  (300 reps)


KeyboardInterrupt: 

In [11]:
# ── Feature-based — FeatureMLP + SVM ─────────────────────────────────────────
feat_subjects = load_feature_subjects_combined(FEAT_TRAIN, FEAT_EVAL, mode='flat_rep')
_feat_per_win = feat_subjects[0][1].shape[2]   # 3192
print(f'Feature input: {feat_subjects[0][1].shape[1]} windows × {_feat_per_win} features/window')

fb_registry = {
    'FeatureMLP': lambda: fb_models.FeatureMLP(input_size=_feat_per_win, dropout_rate=DROPOUT),
}

# MLP — mean-pooled (baseline)
run_within_subject(
    feat_subjects, fb_registry, WS_FB_WEIGHTS, device,
    n_folds=N_FOLDS, batch_size=BATCH_SIZE, seed=SEED,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, lr=LR,
    results_dir=WS_FB_RESULTS,
)

# MLP_W — window-level + majority vote
run_within_subject(
    feat_subjects, fb_registry, WS_FB_WEIGHTS, device,
    n_folds=N_FOLDS, batch_size=BATCH_SIZE, seed=SEED,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, lr=LR,
    results_dir=WS_FB_RESULTS, window_level=True,
)

# SVM — rep-level (mean-pooled, C=50)
run_svm_within_subject(
    feat_subjects, WS_FB_WEIGHTS, WS_FB_RESULTS,
    n_folds=N_FOLDS, seed=SEED,
)

# SVM_W — window-level + majority vote (C=50, mirrors putEMG paper protocol)
run_svm_within_subject(
    feat_subjects, WS_FB_WEIGHTS, WS_FB_RESULTS,
    n_folds=N_FOLDS, seed=SEED, window_level=True,
)


Combining train+eval feature files for 44 subject(s)

  → features_03_flat_rep.npz  X=(272, 26, 3192)
  → features_04_flat_rep.npz  X=(254, 26, 3192)
  → features_05_flat_rep.npz  X=(273, 26, 3192)
  → features_06_flat_rep.npz  X=(300, 26, 3192)
  → features_07_flat_rep.npz  X=(276, 26, 3192)
  → features_08_flat_rep.npz  X=(269, 26, 3192)
  → features_09_flat_rep.npz  X=(273, 26, 3192)
  → features_10_flat_rep.npz  X=(276, 26, 3192)
  → features_11_flat_rep.npz  X=(265, 26, 3192)
  → features_12_flat_rep.npz  X=(279, 26, 3192)
  → features_13_flat_rep.npz  X=(284, 26, 3192)
  → features_14_flat_rep.npz  X=(271, 26, 3192)
  → features_15_flat_rep.npz  X=(274, 26, 3192)
  → features_16_flat_rep.npz  X=(269, 26, 3192)
  → features_17_flat_rep.npz  X=(261, 26, 3192)
  → features_18_flat_rep.npz  X=(278, 26, 3192)
  → features_19_flat_rep.npz  X=(277, 26, 3192)
  → features_20_flat_rep.npz  X=(269, 26, 3192)
  → features_22_flat_rep.npz  X=(281, 26, 3192)
  → features_23_flat_rep.npz  X=(2

{'features_03_flat_rep.npz': [0.7741935483870968,
  0.8444444444444444,
  0.797752808988764],
 'features_04_flat_rep.npz': [0.6091954022988506,
  0.6823529411764706,
  0.6829268292682927],
 'features_05_flat_rep.npz': [0.6413043478260869,
  0.5934065934065934,
  0.5777777777777777],
 'features_06_flat_rep.npz': [0.5673076923076923,
  0.5816326530612245,
  0.46938775510204084],
 'features_07_flat_rep.npz': [0.6, 0.5604395604395604, 0.5777777777777777],
 'features_08_flat_rep.npz': [0.5869565217391305,
  0.5056179775280899,
  0.5340909090909091],
 'features_09_flat_rep.npz': [0.5851063829787234,
  0.6813186813186813,
  0.7954545454545454],
 'features_10_flat_rep.npz': [0.8526315789473684,
  0.7472527472527473,
  0.8222222222222222],
 'features_11_flat_rep.npz': [0.6, 0.48863636363636365, 0.632183908045977],
 'features_12_flat_rep.npz': [0.8247422680412371,
  0.8021978021978022,
  0.8021978021978022],
 'features_13_flat_rep.npz': [0.7346938775510204,
  0.7553191489361702,
  0.684782608695

In [ ]:
# ── Part 2a results summary ───────────────────────────────────────────────────
import glob as _glob

for label, results_dir in [('Deep Learning', WS_DL_RESULTS), ('Feature-Based', WS_FB_RESULTS)]:
    logs = sorted(_glob.glob(os.path.join(results_dir, '*.txt')))
    print(f'\n── {label} ──────────────────────────────────────────────')
    if not logs:
        print('  No result logs yet — run training cells first.')
    else:
        for path in logs:
            with open(path) as f:
                print(f.read())

---
## Part 2b — Cross-Subject Baselines (LOSO)

**Protocol:** Leave-One-Subject-Out — train on 43 subjects, test on the held-out 1 (44 folds).
Val = 10% of training pool (early stopping only; DL/MLP only).

| Model type | Models |
|------------|--------|
| Deep learning | EEGNet, ShallowConvNet, EMG_TCN |
| Feature-based | FeatureMLP (PyTorch) + SVM (sklearn) |

Checkpoints saved per fold — safe to interrupt and resume.

In [ ]:
# ── Part 2b Setup (mount + paths + imports) ───────────────────────────────────
from google.colab import drive
import os, sys, glob, torch
import numpy as np
drive.mount('/content/drive')

_base             = '/content/drive/Othercomputers/My Mac/putEMG prime'
PROCESSED_DIR     = os.path.join(_base, 'data/processed gestures')
TRAIN_DIR         = os.path.join(PROCESSED_DIR, 'train')
EVAL_DIR          = os.path.join(PROCESSED_DIR, 'eval')
FEAT_DIR          = os.path.join(_base, 'data/features')
FEAT_TRAIN        = os.path.join(FEAT_DIR, 'train')
FEAT_EVAL         = os.path.join(FEAT_DIR, 'eval')
CS_DL_WEIGHTS_DIR = os.path.join(_base, 'baselines/cross_subject/deep_learning/weights')
CS_DL_RESULTS_DIR = os.path.join(_base, 'baselines/cross_subject/deep_learning/results')
CS_FB_WEIGHTS_DIR = os.path.join(_base, 'baselines/cross_subject/feature_based/weights')
CS_FB_RESULTS_DIR = os.path.join(_base, 'baselines/cross_subject/feature_based/results')

for _p in (os.path.join(_base, 'src'), os.path.join(_base, 'data_preprocessing')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import deep_learning_models as dl_models
import feature_based_models as fb_models
from emg_loader import load_subjects_combined, load_feature_subjects_combined
from trainer    import run_loso, run_svm_loso

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for _d in (CS_DL_WEIGHTS_DIR, CS_DL_RESULTS_DIR, CS_FB_WEIGHTS_DIR, CS_FB_RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)
print(f'Part 2b setup OK  |  device: {device}')

In [ ]:
# ── LOSO config ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LR            = 1e-3
DROPOUT       = 0.1
PATIENCE      = 5
CS_MAX_EPOCHS = 20
CS_MIN_DELTA  = 0.002
CS_VAL_FRAC   = 0.10

In [ ]:
# ── Deep learning LOSO ────────────────────────────────────────────────────────
dl_subjects  = load_subjects_combined(TRAIN_DIR, EVAL_DIR)
cs_dl_models = {
    'EEGNet':         dl_models.EEGNet,
    'ShallowConvNet': dl_models.ShallowConvNet,
    'EMG_TCN':        dl_models.EMG_TCN,
}

for model_name, model_cls in cs_dl_models.items():
    print(f'\n{{"="*60}}')
    print(f'  DL LOSO — {{model_name}}')
    print(f'{{"="*60}}')
    run_loso(
        subjects    = dl_subjects,
        model_cls   = model_cls,
        model_type  = model_name,
        weights_dir = os.path.join(CS_DL_WEIGHTS_DIR, model_name),
        log_path    = os.path.join(CS_DL_RESULTS_DIR, f'results_log_{{model_name}}.txt'),
        device      = device,
        val_frac    = CS_VAL_FRAC,
        batch_size  = BATCH_SIZE,
        dropout     = DROPOUT,
        lr          = LR,
        max_epochs  = CS_MAX_EPOCHS,
        patience    = PATIENCE,
        min_delta   = CS_MIN_DELTA,
    )

In [ ]:
# ── Feature-based LOSO — FeatureMLP + SVM ────────────────────────────────────
cs_feat_subjects = load_feature_subjects_combined(FEAT_TRAIN, FEAT_EVAL, mode='flat_rep')
_feat_per_win    = cs_feat_subjects[0][1].shape[2]   # 3192
print(f'Feature input: {cs_feat_subjects[0][1].shape[1]} windows × {_feat_per_win} features/window')

# MLP LOSO
cs_fb_models = {
    'FeatureMLP': lambda dropout_rate: fb_models.FeatureMLP(
                      input_size=_feat_per_win, dropout_rate=dropout_rate),
}
for model_name, model_cls in cs_fb_models.items():
    print(f'\n{"="*60}\n  Feature LOSO — {model_name}\n{"="*60}')
    run_loso(
        subjects    = cs_feat_subjects,
        model_cls   = model_cls,
        model_type  = model_name,
        weights_dir = os.path.join(CS_FB_WEIGHTS_DIR, model_name),
        log_path    = os.path.join(CS_FB_RESULTS_DIR, f'results_log_{model_name}.txt'),
        device      = device,
        val_frac    = CS_VAL_FRAC,
        batch_size  = BATCH_SIZE,
        dropout     = DROPOUT,
        lr          = LR,
        max_epochs  = CS_MAX_EPOCHS,
        patience    = PATIENCE,
        min_delta   = CS_MIN_DELTA,
    )

# SVM LOSO — rep-level (mean-pooled, C=50)
print(f'\n{"="*60}\n  Feature LOSO — SVM\n{"="*60}')
run_svm_loso(
    subjects    = cs_feat_subjects,
    weights_dir = os.path.join(CS_FB_WEIGHTS_DIR, 'SVM'),
    log_path    = os.path.join(CS_FB_RESULTS_DIR, 'results_log_SVM.txt'),
)

# SVM_W LOSO — window-level + majority vote (C=50, mirrors putEMG paper protocol)
print(f'\n{"="*60}\n  Feature LOSO — SVM_W\n{"="*60}')
run_svm_loso(
    subjects    = cs_feat_subjects,
    weights_dir = os.path.join(CS_FB_WEIGHTS_DIR, 'SVM_W'),
    log_path    = os.path.join(CS_FB_RESULTS_DIR, 'results_log_SVM_W.txt'),
    window_level = True,
)


In [ ]:
# ── Part 2b results summary ───────────────────────────────────────────────────
import glob as _glob

for label, results_dir in [
    ('Deep Learning',  CS_DL_RESULTS_DIR),
    ('Feature-Based',  CS_FB_RESULTS_DIR),
]:
    logs = sorted(_glob.glob(os.path.join(results_dir, 'results_log_*.txt')))
    print(f'\n── {{label}} LOSO ──────────────────────────────────────')
    if not logs:
        print('  No logs yet — run training cells first.')
    else:
        for path in logs:
            print(f'\n  {os.path.basename(path)}')
            with open(path) as f:
                print(f.read())